In [ ]:
from google.colab import files

uploaded = files.upload()

Saving Rappo qa.pdf to Rappo qa.pdf


In [ ]:
# !pip install streamlit langchain faiss-cpu sentence-transformers PyPDF2 nltk requests pandas numpy langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.1/438.1 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.0/363.0 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.7 MB/s eta 0:00:00
  Attempting uninstall: langsmith
    Found existing installation: langsmith 0.3.44
    Uninstalling langsmith-0.3.44:
      Successfully uninstalled langsmith-0.3.44
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.63
    Uninstalling langchain-core-0.3.63:
      Successfully uninstalled langchain-core-0.3.63


In [ ]:
# Cell 1: Install and Import

import os
import pandas as pd
import numpy as np
from typing import List, Dict, Any, Tuple
import time
from datetime import datetime
import requests
import json
import streamlit as st

# Document processing
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.retrievers import BM25Retriever, EnsembleRetriever
from langchain.schema import Document
from langchain.llms.base import LLM
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

import nltk
try:
    nltk.download('punkt', quiet=True)
    nltk.download('stopwords', quiet=True)
except:
    pass

In [ ]:
# Cell 2: Set up API Key
GROQ_API_KEY = 'Your grok key here'

# Custom Groq LLM class
class GroqLLM(LLM):
    def __init__(self, api_key: str, model: str = "llama3-8b-8192"):
        super().__init__()
        self._api_key = api_key
        self._model = model
        self._base_url = "https://api.groq.com/openai/v1/chat/completions"

    @property
    def _llm_type(self) -> str:
        return "groq"

    def _call(self, prompt: str, stop=None, run_manager=None, **kwargs) -> str:
        headers = {
            "Authorization": f"Bearer {self._api_key}",
            "Content-Type": "application/json"
        }

        data = {
            "model": self._model,
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0.5,
            "max_tokens": 500
        }

        try:
            response = requests.post(self._base_url, headers=headers, json=data)
            response.raise_for_status()
            result = response.json()
            return result["choices"][0]["message"]["content"]
        except Exception as e:
            return f"Error: {str(e)}"

In [ ]:
# Cell 3: Enhanced PDF FAQ System with Processing Time and Confidence Score
class PDFDocumentFAQSystem:
    def __init__(self):
        self.embeddings = HuggingFaceEmbeddings(
            model_name='sentence-transformers/all-MiniLM-L6-v2'
        )

        groq_api_key = GROQ_API_KEY
        if not groq_api_key or groq_api_key == 'YOUR_GROQ_API_KEY_HERE':
            raise ValueError("Please set your GROQ API key in the GROQ_API_KEY variable!")

        self.llm = GroqLLM(api_key=groq_api_key, model='llama3-8b-8192')

        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=1000,
            chunk_overlap=200,
            length_function=len
        )

        self.documents = []
        self.vectorstore = None
        self.qa_chain = None
        self.retriever = None

    def load_pdf(self, file_path: str) -> bool:
        try:
            if not os.path.exists(file_path):
                print(f"File not found: {file_path}")
                return False

            if not file_path.lower().endswith('.pdf'):
                print("Please provide a PDF file.")
                return False

            print("Loading PDF...")
            loader = PyPDFLoader(file_path)
            documents = loader.load()

            print("Processing document...")
            chunks = self.text_splitter.split_documents(documents)

            print("Creating embeddings...")
            self.vectorstore = FAISS.from_documents(chunks, self.embeddings)

            semantic_retriever = self.vectorstore.as_retriever(search_kwargs={"k": 5})
            texts = [doc.page_content for doc in chunks]
            bm25_retriever = BM25Retriever.from_texts(texts)
            bm25_retriever.k = 5

            self.retriever = EnsembleRetriever(
                retrievers=[semantic_retriever, bm25_retriever],
                weights=[0.7, 0.3]
            )

            prompt_template = """
            Use the following context to answer the question accurately.

            Context: {context}
            Question: {question}

            Provide a clear, accurate answer based on the context. If the context doesn't contain enough information, say so clearly.

            Answer:"""

            PROMPT = PromptTemplate(
                template=prompt_template,
                input_variables=["context", "question"]
            )

            self.qa_chain = RetrievalQA.from_chain_type(
                llm=self.llm,
                chain_type="stuff",
                retriever=self.retriever,
                chain_type_kwargs={"prompt": PROMPT},
                return_source_documents=True
            )

            print("PDF processed successfully!")
            return True

        except Exception as e:
            print(f"Error processing PDF: {str(e)}")
            return False

    def calculate_confidence_score(self, question: str, answer: str, source_docs: List[Document]) -> float:
        try:
            doc_factor = min(len(source_docs) / 5.0, 1.0)

            if self.vectorstore and question:
                similar_docs = self.vectorstore.similarity_search_with_score(question, k=3)
                if similar_docs:
                    avg_similarity = np.mean([1 / (1 + score) for _, score in similar_docs])
                    similarity_factor = min(avg_similarity, 1.0)
                else:
                    similarity_factor = 0.5
            else:
                similarity_factor = 0.5

            answer_length = len(answer.split())
            length_factor = min(answer_length / 50.0, 1.0)

            uncertainty_phrases = [
                "i don't know", "not sure", "unclear", "cannot determine",
                "insufficient information", "not enough information", "doesn't contain"
            ]
            uncertainty_penalty = 0.3 if any(phrase in answer.lower() for phrase in uncertainty_phrases) else 0

            confidence = (doc_factor * 0.3 + similarity_factor * 0.4 + length_factor * 0.3) - uncertainty_penalty
            confidence = max(0.1, min(1.0, confidence))

            return round(confidence, 2)

        except Exception as e:
            print(f"Error calculating confidence: {str(e)}")
            return 0.5

    def ask_question(self, question: str) -> Tuple[str, float, float]:
        if not self.qa_chain:
            return "Please load a PDF document first.", 0.0, 0.0

        start_time = time.time()

        try:
            result = self.qa_chain({"query": question})
            processing_time = time.time() - start_time

            answer = result['result']
            source_docs = result.get('source_documents', [])

            confidence = self.calculate_confidence_score(question, answer, source_docs)

            return answer, processing_time, confidence

        except Exception as e:
            processing_time = time.time() - start_time
            return f"Error: {str(e)}", processing_time, 0.0

In [ ]:
# Cell 4: Command Line Interface
def main_cli():
    try:
        faq_system = PDFDocumentFAQSystem()
    except ValueError as e:
        print(f"Error: {e}")
        return

    while True:
        file_path = input("Enter PDF file path: ").strip()
        if file_path:
            if faq_system.load_pdf(file_path):
                break
            else:
                print("Please try again with a valid PDF file path.")
        else:
            print("Please enter a file path.")

    print("\nYou can now ask questions about the PDF. Type 'quit' to exit.")

    while True:
        question = input("\nYour question: ").strip()

        if question.lower() in ['quit', 'exit', 'q']:
            print("Goodbye!")
            break

        if question:
            print("\nProcessing...")
            answer, processing_time, confidence = faq_system.ask_question(question)

            print(f"\nAnswer: {answer}")
            print(f"Processing Time: {processing_time:.2f} seconds")
            print(f"Confidence Score: {confidence:.2f}/1.00")
        else:
            print("Please enter a question.")

In [ ]:
# Cell 5: Streamlit App
def main_streamlit():
    st.set_page_config(
        page_title="PDF FAQ System",
        page_icon="📄",
        layout="wide"
    )

    st.title("📄 PDF Document FAQ System")
    st.markdown("Upload a PDF document and ask questions about its content!")

    if 'faq_system' not in st.session_state:
        try:
            st.session_state.faq_system = PDFDocumentFAQSystem()
            st.session_state.pdf_loaded = False
        except ValueError as e:
            st.error(f"Error initializing system: {e}")
            st.stop()

    with st.sidebar:
        st.header("📁 Document Upload")

        pdf_path = st.text_input("Enter PDF file path:", placeholder="/content/your_file.pdf")

        if st.button("Load PDF") and pdf_path:
            if not st.session_state.pdf_loaded:
                with st.spinner("Processing PDF... This may take a moment."):
                    success = st.session_state.faq_system.load_pdf(pdf_path)

                if success:
                    st.session_state.pdf_loaded = True
                    st.success("✅ PDF loaded successfully!")
                else:
                    st.error("❌ Failed to load PDF. Please try again.")

    if st.session_state.pdf_loaded:
        st.success("📖 PDF document is ready for questions!")

        col1, col2 = st.columns([2, 1])

        with col1:
            question = st.text_area(
                "Ask a question about the PDF:",
                height=100,
                placeholder="Enter your question here..."
            )

            ask_button = st.button("🔍 Ask Question", type="primary")

        with col2:
            st.markdown("### 📊 Metrics")
            time_placeholder = st.empty()
            confidence_placeholder = st.empty()

        if ask_button and question:
            with st.spinner("Processing your question..."):
                answer, processing_time, confidence = st.session_state.faq_system.ask_question(question)

            time_placeholder.metric("⏱️ Processing Time", f"{processing_time:.2f}s")
            confidence_placeholder.metric("🎯 Confidence Score", f"{confidence:.2f}/1.00")

            st.markdown("### 💬 Answer")

            if confidence >= 0.8:
                st.success(answer)
            elif confidence >= 0.6:
                st.info(answer)
            elif confidence >= 0.4:
                st.warning(answer)
            else:
                st.error(answer)

            st.markdown("### 📈 Confidence Interpretation")
            if confidence >= 0.8:
                st.markdown("🟢 **High Confidence** - The answer is likely very accurate based on the document content.")
            elif confidence >= 0.6:
                st.markdown("🟡 **Medium Confidence** - The answer is reasonably accurate but may need verification.")
            elif confidence >= 0.4:
                st.markdown("🟠 **Low Confidence** - The answer is uncertain, please verify with the original document.")
            else:
                st.markdown("🔴 **Very Low Confidence** - The information may not be available in the document.")

        elif ask_button and not question:
            st.warning("Please enter a question first.")

    else:
        st.info("👈 Please enter PDF file path and click 'Load PDF' to get started!")

        st.markdown("### 💡 How to use:")
        st.markdown("""
        1. **Enter PDF Path**: Use the sidebar to enter your PDF file path (e.g., /content/your_file.pdf)
        2. **Load PDF**: Click 'Load PDF' button to process your document
        3. **Ask Questions**: Type your questions in the text area
        4. **Get Answers**: View answers with processing time and confidence scores
        """)


In [ ]:
# Cell 6: Run Streamlit in Colab
%%writefile streamlit_app.py

import sys
sys.path.append('/content')

import os
import pandas as pd
import numpy as np
from typing import List, Dict, Any, Tuple
import time
from datetime import datetime
import requests
import json
import streamlit as st

from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.retrievers import BM25Retriever, EnsembleRetriever
from langchain.schema import Document
from langchain.llms.base import LLM
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

import nltk
try:
    nltk.download('punkt', quiet=True)
    nltk.download('stopwords', quiet=True)
except:
    pass

GROQ_API_KEY = 'Your grok key here'

class GroqLLM(LLM):
    def __init__(self, api_key: str, model: str = "llama3-8b-8192"):
        super().__init__()
        self._api_key = api_key
        self._model = model
        self._base_url = "https://api.groq.com/openai/v1/chat/completions"

    @property
    def _llm_type(self) -> str:
        return "groq"

    def _call(self, prompt: str, stop=None, run_manager=None, **kwargs) -> str:
        headers = {
            "Authorization": f"Bearer {self._api_key}",
            "Content-Type": "application/json"
        }

        data = {
            "model": self._model,
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0.1,
            "max_tokens": 500
        }

        try:
            response = requests.post(self._base_url, headers=headers, json=data)
            response.raise_for_status()
            result = response.json()
            return result["choices"][0]["message"]["content"]
        except Exception as e:
            return f"Error: {str(e)}"

class PDFDocumentFAQSystem:
    def __init__(self):
        self.embeddings = HuggingFaceEmbeddings(
            model_name='sentence-transformers/all-MiniLM-L6-v2'
        )

        groq_api_key = GROQ_API_KEY
        if not groq_api_key or groq_api_key == 'YOUR_GROQ_API_KEY_HERE':
            raise ValueError("Please set your GROQ API key in the GROQ_API_KEY variable!")

        self.llm = GroqLLM(api_key=groq_api_key, model='llama3-8b-8192')

        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=1000,
            chunk_overlap=200,
            length_function=len
        )

        self.documents = []
        self.vectorstore = None
        self.qa_chain = None
        self.retriever = None

    def load_pdf(self, file_path: str) -> bool:
        try:
            if not os.path.exists(file_path):
                print(f"File not found: {file_path}")
                return False

            if not file_path.lower().endswith('.pdf'):
                print("Please provide a PDF file.")
                return False

            print("Loading PDF...")
            loader = PyPDFLoader(file_path)
            documents = loader.load()

            print("Processing document...")
            chunks = self.text_splitter.split_documents(documents)

            print("Creating embeddings...")
            self.vectorstore = FAISS.from_documents(chunks, self.embeddings)

            semantic_retriever = self.vectorstore.as_retriever(search_kwargs={"k": 5})
            texts = [doc.page_content for doc in chunks]
            bm25_retriever = BM25Retriever.from_texts(texts)
            bm25_retriever.k = 5

            self.retriever = EnsembleRetriever(
                retrievers=[semantic_retriever, bm25_retriever],
                weights=[0.7, 0.3]
            )

            prompt_template = """
            Use the following context to answer the question accurately.

            Context: {context}
            Question: {question}

            Provide a clear, accurate answer based on the context. If the context doesn't contain enough information, say so clearly.

            Answer:"""

            PROMPT = PromptTemplate(
                template=prompt_template,
                input_variables=["context", "question"]
            )

            self.qa_chain = RetrievalQA.from_chain_type(
                llm=self.llm,
                chain_type="stuff",
                retriever=self.retriever,
                chain_type_kwargs={"prompt": PROMPT},
                return_source_documents=True
            )

            print("PDF processed successfully!")
            return True

        except Exception as e:
            print(f"Error processing PDF: {str(e)}")
            return False

    def calculate_confidence_score(self, question: str, answer: str, source_docs: List[Document]) -> float:
        try:
            doc_factor = min(len(source_docs) / 5.0, 1.0)

            if self.vectorstore and question:
                similar_docs = self.vectorstore.similarity_search_with_score(question, k=3)
                if similar_docs:
                    avg_similarity = np.mean([1 / (1 + score) for _, score in similar_docs])
                    similarity_factor = min(avg_similarity, 1.0)
                else:
                    similarity_factor = 0.5
            else:
                similarity_factor = 0.5

            answer_length = len(answer.split())
            length_factor = min(answer_length / 50.0, 1.0)

            uncertainty_phrases = [
                "i don't know", "not sure", "unclear", "cannot determine",
                "insufficient information", "not enough information", "doesn't contain"
            ]
            uncertainty_penalty = 0.3 if any(phrase in answer.lower() for phrase in uncertainty_phrases) else 0

            confidence = (doc_factor * 0.3 + similarity_factor * 0.4 + length_factor * 0.3) - uncertainty_penalty
            confidence = max(0.1, min(1.0, confidence))

            return round(confidence, 2)

        except Exception as e:
            print(f"Error calculating confidence: {str(e)}")
            return 0.5

    def ask_question(self, question: str) -> Tuple[str, float, float]:
        if not self.qa_chain:
            return "Please load a PDF document first.", 0.0, 0.0

        start_time = time.time()

        try:
            result = self.qa_chain({"query": question})
            processing_time = time.time() - start_time

            answer = result['result']
            source_docs = result.get('source_documents', [])

            confidence = self.calculate_confidence_score(question, answer, source_docs)

            return answer, processing_time, confidence

        except Exception as e:
            processing_time = time.time() - start_time
            return f"Error: {str(e)}", processing_time, 0.0

def main_streamlit():
    st.set_page_config(
        page_title="PDF FAQ System",
        page_icon="📄",
        layout="wide"
    )

    st.title("📄 PDF Document FAQ System")
    st.markdown("Upload a PDF document and ask questions about its content!")

    if 'faq_system' not in st.session_state:
        try:
            st.session_state.faq_system = PDFDocumentFAQSystem()
            st.session_state.pdf_loaded = False
        except ValueError as e:
            st.error(f"Error initializing system: {e}")
            st.stop()

    with st.sidebar:
        st.header("📁 Document Upload")

        pdf_path = st.text_input("Enter PDF file path:", placeholder="/content/your_file.pdf")

        if st.button("Load PDF") and pdf_path:
            if not st.session_state.pdf_loaded:
                with st.spinner("Processing PDF... This may take a moment."):
                    success = st.session_state.faq_system.load_pdf(pdf_path)

                if success:
                    st.session_state.pdf_loaded = True
                    st.success("✅ PDF loaded successfully!")
                else:
                    st.error("❌ Failed to load PDF. Please try again.")

    if st.session_state.pdf_loaded:
        st.success("📖 PDF document is ready for questions!")

        col1, col2 = st.columns([2, 1])

        with col1:
            question = st.text_area(
                "Ask a question about the PDF:",
                height=100,
                placeholder="Enter your question here..."
            )

            ask_button = st.button("🔍 Ask Question", type="primary")

        with col2:
            st.markdown("### 📊 Metrics")
            time_placeholder = st.empty()
            confidence_placeholder = st.empty()

        if ask_button and question:
            with st.spinner("Processing your question..."):
                answer, processing_time, confidence = st.session_state.faq_system.ask_question(question)

            time_placeholder.metric("⏱️ Processing Time", f"{processing_time:.2f}s")
            confidence_placeholder.metric("🎯 Confidence Score", f"{confidence:.2f}/1.00")

            st.markdown("### 💬 Answer")

            if confidence >= 0.8:
                st.success(answer)
            elif confidence >= 0.6:
                st.info(answer)
            elif confidence >= 0.4:
                st.warning(answer)
            else:
                st.error(answer)

            st.markdown("### 📈 Confidence Interpretation")
            if confidence >= 0.8:
                st.markdown("🟢 **High Confidence** - The answer is likely very accurate based on the document content.")
            elif confidence >= 0.6:
                st.markdown("🟡 **Medium Confidence** - The answer is reasonably accurate but may need verification.")
            elif confidence >= 0.4:
                st.markdown("🟠 **Low Confidence** - The answer is uncertain, please verify with the original document.")
            else:
                st.markdown("🔴 **Very Low Confidence** - The information may not be available in the document.")

        elif ask_button and not question:
            st.warning("Please enter a question first.")

    else:
        st.info("👈 Please enter PDF file path and click 'Load PDF' to get started!")

        st.markdown("### 💡 How to use:")
        st.markdown("""
        1. **Enter PDF Path**: Use the sidebar to enter your PDF file path (e.g., /content/your_file.pdf)
        2. **Load PDF**: Click 'Load PDF' button to process your document
        3. **Ask Questions**: Type your questions in the text area
        4. **Get Answers**: View answers with processing time and confidence scores
        """)

if __name__ == "__main__":
    main_streamlit()

Writing streamlit_app.py


In [ ]:
# Cell 7: Start Streamlit with Colab Tunnel
# !pip install pyngrok

import subprocess
import threading
from pyngrok import ngrok
import time
import os

# Kill any existing streamlit processes
!pkill -f streamlit

# Add your ngrok authtoken here
# Replace 'YOUR_NGROK_AUTHTOKEN' with your actual authtoken from https://dashboard.ngrok.com/get-started/your-authtoken
# It's recommended to store this as an environment variable for security in production
# *** REPLACE 'YOUR_NGROK_AUTHTOKEN' BELOW WITH YOUR ACTUAL NGROK AUTH TOKEN ***
NGROK_AUTHTOKEN = os.environ.get("NGROK_AUTHTOKEN", "2yJey6nB4wBpzBrxy4RHlRaTbsX_3CW538mncRysQJvFgXf8n") # Replace with your token or set environment variable
ngrok.set_auth_token(NGROK_AUTHTOKEN)

# Start Streamlit in background
def run_streamlit():
    subprocess.run(['streamlit', 'run', 'streamlit_app.py', '--server.port', '8501', '--server.headless', 'true'])

# Start streamlit in a separate thread
streamlit_thread = threading.Thread(target=run_streamlit)
streamlit_thread.daemon = True
streamlit_thread.start()

# Wait a moment for streamlit to start
time.sleep(10)

In [ ]:

# Cell 8: Create ngrok tunnel and keep alive

# Create ngrok tunnel
public_url = ngrok.connect(8501)
print(f"🌐 Your Streamlit app is running at: {public_url}")
print(f"📱 Click the link above to access your PDF FAQ System!")

# Keep the tunnel alive
try:
    # This will keep the cell running and the tunnel active
    # Wait indefinitely for the streamlit thread to finish
    streamlit_thread.join()
except KeyboardInterrupt:
    print("Stopping the application...")
finally:
    # Ensure ngrok process is killed on exit
    ngrok.kill()

🌐 Your Streamlit app is running at: NgrokTunnel: "https://07bd-35-237-69-85.ngrok-free.app" -> "http://localhost:8501"
📱 Click the link above to access your PDF FAQ System!


In [ ]:
# !pip install pypdf
# !pip install rank_bm25

In [ ]:



# Cell 8: Alternative - Direct CLI
main_cli()

Enter PDF file path: /content/Rappo qa.pdf
Loading PDF...
Processing document...
Creating embeddings...
PDF processed successfully!

You can now ask questions about the PDF. Type 'quit' to exit.

Your question: what is thi pdf about?

Processing...


<ipython-input-7-3063286371>:124: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result = self.qa_chain({"query": question})



Answer: Based on the provided context, this PDF appears to be about Rappo, a marketplace that connects early-stage startups with key decision-makers in larger companies. The PDF provides information on how Rappo works, the benefits of using the platform, and the roles and responsibilities of both startups and champions (industry experts) on the platform. It also addresses questions and concerns about using Rappo, including costs, confidentiality, and conflict of interest.
Processing Time: 0.59 seconds
Confidence Score: 0.76/1.00

Your question: How many champions are present at present at rappo ?

Processing...

Answer: Unfortunately, the context does not provide information on the number of champions present at Rappo. The context only mentions the attributes of a champion, their role, and the benefits of becoming a champion, but it does not provide a specific number. Therefore, I cannot provide an accurate answer to this question.
Processing Time: 0.50 seconds
Confidence Score: 0.75/

In [ ]:
!pip install streamlit

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 51.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 6.6 MB/s eta 0:00:00


In [ ]:
# # Cell 1: Install and Import
# !pip install streamlit langchain faiss-cpu sentence-transformers PyPDF2 nltk requests pandas numpy

# import os
# import pandas as pd
# import numpy as np
# from typing import List, Dict, Any, Tuple
# import time
# from datetime import datetime
# import requests
# import json
# import streamlit as st

# # Document processing
# from langchain.document_loaders import PyPDFLoader
# from langchain.text_splitter import RecursiveCharacterTextSplitter
# from langchain.embeddings import HuggingFaceEmbeddings
# from langchain.vectorstores import FAISS
# from langchain.retrievers import BM25Retriever, EnsembleRetriever
# from langchain.schema import Document
# from langchain.llms.base import LLM
# from langchain.chains import RetrievalQA
# from langchain.prompts import PromptTemplate

# import nltk
# try:
#     nltk.download('punkt', quiet=True)
#     nltk.download('stopwords', quiet=True)
# except:
#     pass

# # Cell 2: Set up API Key
# GROQ_API_KEY = 'YOUR_GROQ_API_KEY_HERE' #Replace with your API Key

# # Custom Groq LLM class
# class GroqLLM(LLM):
#     def __init__(self, api_key: str, model: str = "llama3-8b-8192"):
#         super().__init__()
#         self._api_key = api_key
#         self._model = model
#         self._base_url = "https://api.groq.com/openai/v1/chat/completions"

#     @property
#     def _llm_type(self) -> str:
#         return "groq"

#     def _call(self, prompt: str, stop=None, run_manager=None, **kwargs) -> str:
#         headers = {
#             "Authorization": f"Bearer {self._api_key}",
#             "Content-Type": "application/json"
#         }

#         data = {
#             "model": self._model,
#             "messages": [{"role": "user", "content": prompt}],
#             "temperature": 0.1,
#             "max_tokens": 500
#         }

#         try:
#             response = requests.post(self._base_url, headers=headers, json=data)
#             response.raise_for_status()
#             result = response.json()
#             return result["choices"][0]["message"]["content"]
#         except Exception as e:
#             return f"Error: {str(e)}"

# # Cell 3: Enhanced PDF FAQ System with Processing Time and Confidence Score
# class PDFDocumentFAQSystem:
#     def __init__(self):
#         self.embeddings = HuggingFaceEmbeddings(
#             model_name='sentence-transformers/all-MiniLM-L6-v2'
#         )

#         groq_api_key = GROQ_API_KEY
#         if not groq_api_key or groq_api_key == 'YOUR_GROQ_API_KEY_HERE':
#             raise ValueError("Please set your GROQ API key in the GROQ_API_KEY variable!")

#         self.llm = GroqLLM(api_key=groq_api_key, model='llama3-8b-8192')

#         self.text_splitter = RecursiveCharacterTextSplitter(
#             chunk_size=1000,
#             chunk_overlap=200,
#             length_function=len
#         )

#         self.documents = []
#         self.vectorstore = None
#         self.qa_chain = None
#         self.retriever = None

#     def load_pdf(self, file_path: str) -> bool:
#         try:
#             if not os.path.exists(file_path):
#                 print(f"File not found: {file_path}")
#                 return False

#             if not file_path.lower().endswith('.pdf'):
#                 print("Please provide a PDF file.")
#                 return False

#             print("Loading PDF...")
#             loader = PyPDFLoader(file_path)
#             documents = loader.load()

#             print("Processing document...")
#             chunks = self.text_splitter.split_documents(documents)

#             print("Creating embeddings...")
#             self.vectorstore = FAISS.from_documents(chunks, self.embeddings)

#             semantic_retriever = self.vectorstore.as_retriever(search_kwargs={"k": 5})
#             texts = [doc.page_content for doc in chunks]
#             bm25_retriever = BM25Retriever.from_texts(texts)
#             bm25_retriever.k = 5

#             self.retriever = EnsembleRetriever(
#                 retrievers=[semantic_retriever, bm25_retriever],
#                 weights=[0.7, 0.3]
#             )

#             prompt_template = """
#             Use the following context to answer the question accurately.

#             Context: {context}
#             Question: {question}

#             Provide a clear, accurate answer based on the context. If the context doesn't contain enough information, say so clearly.

#             Answer:"""

#             PROMPT = PromptTemplate(
#                 template=prompt_template,
#                 input_variables=["context", "question"]
#             )

#             self.qa_chain = RetrievalQA.from_chain_type(
#                 llm=self.llm,
#                 chain_type="stuff",
#                 retriever=self.retriever,
#                 chain_type_kwargs={"prompt": PROMPT},
#                 return_source_documents=True
#             )

#             print("PDF processed successfully!")
#             return True

#         except Exception as e:
#             print(f"Error processing PDF: {str(e)}")
#             return False

#     def calculate_confidence_score(self, question: str, answer: str, source_docs: List[Document]) -> float:
#         try:
#             doc_factor = min(len(source_docs) / 5.0, 1.0)

#             if self.vectorstore and question:
#                 similar_docs = self.vectorstore.similarity_search_with_score(question, k=3)
#                 if similar_docs:
#                     avg_similarity = np.mean([1 / (1 + score) for _, score in similar_docs])
#                     similarity_factor = min(avg_similarity, 1.0)
#                 else:
#                     similarity_factor = 0.5
#             else:
#                 similarity_factor = 0.5

#             answer_length = len(answer.split())
#             length_factor = min(answer_length / 50.0, 1.0)

#             uncertainty_phrases = [
#                 "i don't know", "not sure", "unclear", "cannot determine",
#                 "insufficient information", "not enough information", "doesn't contain"
#             ]
#             uncertainty_penalty = 0.3 if any(phrase in answer.lower() for phrase in uncertainty_phrases) else 0

#             confidence = (doc_factor * 0.3 + similarity_factor * 0.4 + length_factor * 0.3) - uncertainty_penalty
#             confidence = max(0.1, min(1.0, confidence))

#             return round(confidence, 2)

#         except Exception as e:
#             print(f"Error calculating confidence: {str(e)}")
#             return 0.5

#     def ask_question(self, question: str) -> Tuple[str, float, float]:
#         if not self.qa_chain:
#             return "Please load a PDF document first.", 0.0, 0.0

#         start_time = time.time()

#         try:
#             result = self.qa_chain({"query": question})
#             processing_time = time.time() - start_time

#             answer = result['result']
#             source_docs = result.get('source_documents', [])

#             confidence = self.calculate_confidence_score(question, answer, source_docs)

#             return answer, processing_time, confidence

#         except Exception as e:
#             processing_time = time.time() - start_time
#             return f"Error: {str(e)}", processing_time, 0.0

# # Cell 4: Command Line Interface (Enhanced)
# def main_cli():
#     try:
#         faq_system = PDFDocumentFAQSystem()
#     except ValueError as e:
#         print(f"Error: {e}")
#         return

#     while True:
#         file_path = input("Enter PDF file path: ").strip()
#         if file_path:
#             if faq_system.load_pdf(file_path):
#                 break
#             else:
#                 print("Please try again with a valid PDF file path.")
#         else:
#             print("Please enter a file path.")

#     print("\nYou can now ask questions about the PDF. Type 'quit' to exit.")

#     while True:
#         question = input("\nYour question: ").strip()

#         if question.lower() in ['quit', 'exit', 'q']:
#             print("Goodbye!")
#             break

#         if question:
#             print("\nProcessing...")
#             answer, processing_time, confidence = faq_system.ask_question(question)

#             print(f"\nAnswer: {answer}")
#             print(f"Processing Time: {processing_time:.2f} seconds")
#             print(f"Confidence Score: {confidence:.2f}/1.00")
#         else:
#             print("Please enter a question.")

# # Cell 5: Streamlit App
# def main_streamlit():
#     st.set_page_config(
#         page_title="PDF FAQ System",
#         page_icon="📄",
#         layout="wide"
#     )

#     st.title("📄 PDF Document FAQ System")
#     st.markdown("Upload a PDF document and ask questions about its content!")

#     if 'faq_system' not in st.session_state:
#         try:
#             st.session_state.faq_system = PDFDocumentFAQSystem()
#             st.session_state.pdf_loaded = False
#         except ValueError as e:
#             st.error(f"Error initializing system: {e}")
#             st.stop()

#     with st.sidebar:
#         st.header("📁 Document Upload")

#         pdf_path = st.text_input("Enter PDF file path:", placeholder="/content/your_file.pdf")

#         if st.button("Load PDF") and pdf_path:
#             if not st.session_state.pdf_loaded:
#                 with st.spinner("Processing PDF... This may take a moment."):
#                     success = st.session_state.faq_system.load_pdf(pdf_path)

#                 if success:
#                     st.session_state.pdf_loaded = True
#                     st.success("✅ PDF loaded successfully!")
#                 else:
#                     st.error("❌ Failed to load PDF. Please try again.")

#     if st.session_state.pdf_loaded:
#         st.success("📖 PDF document is ready for questions!")

#         col1, col2 = st.columns([2, 1])

#         with col1:
#             question = st.text_area(
#                 "Ask a question about the PDF:",
#                 height=100,
#                 placeholder="Enter your question here..."
#             )

#             ask_button = st.button("🔍 Ask Question", type="primary")

#         with col2:
#             st.markdown("### 📊 Metrics")
#             time_placeholder = st.empty()
#             confidence_placeholder = st.empty()

#         if ask_button and question:
#             with st.spinner("Processing your question..."):
#                 answer, processing_time, confidence = st.session_state.faq_system.ask_question(question)

#             time_placeholder.metric("⏱️ Processing Time", f"{processing_time:.2f}s")
#             confidence_placeholder.metric("🎯 Confidence Score", f"{confidence:.2f}/1.00")

#             st.markdown("### 💬 Answer")

#             if confidence >= 0.8:
#                 st.success(answer)
#             elif confidence >= 0.6:
#                 st.info(answer)
#             elif confidence >= 0.4:
#                 st.warning(answer)
#             else:
#                 st.error(answer)

#             st.markdown("### 📈 Confidence Interpretation")
#             if confidence >= 0.8:
#                 st.markdown("🟢 **High Confidence** - The answer is likely very accurate based on the document content.")
#             elif confidence >= 0.6:
#                 st.markdown("🟡 **Medium Confidence** - The answer is reasonably accurate but may need verification.")
#             elif confidence >= 0.4:
#                 st.markdown("🟠 **Low Confidence** - The answer is uncertain, please verify with the original document.")
#             else:
#                 st.markdown("🔴 **Very Low Confidence** - The information may not be available in the document.")

#         elif ask_button and not question:
#             st.warning("Please enter a question first.")

#     else:
#         st.info("👈 Please enter PDF file path and click 'Load PDF' to get started!")

#         st.markdown("### 💡 How to use:")
#         st.markdown("""
#         1. **Enter PDF Path**: Use the sidebar to enter your PDF file path (e.g., /content/your_file.pdf)
#         2. **Load PDF**: Click 'Load PDF' button to process your document
#         3. **Ask Questions**: Type your questions in the text area
#         4. **Get Answers**: View answers with processing time and confidence scores
#         """)

# # Cell 6: Run Streamlit in Colab
# %%writefile streamlit_app.py

# import sys
# sys.path.append('/content')

# import os
# import pandas as pd
# import numpy as np
# from typing import List, Dict, Any, Tuple
# import time
# from datetime import datetime
# import requests
# import json
# import streamlit as st

# from langchain.document_loaders import PyPDFLoader
# from langchain.text_splitter import RecursiveCharacterTextSplitter
# from langchain.embeddings import HuggingFaceEmbeddings
# from langchain.vectorstores import FAISS
# from langchain.retrievers import BM25Retriever, EnsembleRetriever
# from langchain.schema import Document
# from langchain.llms.base import LLM
# from langchain.chains import RetrievalQA
# from langchain.prompts import PromptTemplate

# import nltk
# try:
#     nltk.download('punkt', quiet=True)
#     nltk.download('stopwords', quiet=True)
# except:
#     pass

# GROQ_API_KEY = 'YOUR_GROQ_API_KEY_HERE'

# class GroqLLM(LLM):
#     def __init__(self, api_key: str, model: str = "llama3-8b-8192"):
#         super().__init__()
#         self._api_key = api_key
#         self._model = model
#         self._base_url = "https://api.groq.com/openai/v1/chat/completions"

#     @property
#     def _llm_type(self) -> str:
#         return "groq"

#     def _call(self, prompt: str, stop=None, run_manager=None, **kwargs) -> str:
#         headers = {
#             "Authorization": f"Bearer {self._api_key}",
#             "Content-Type": "application/json"
#         }

#         data = {
#             "model": self._model,
#             "messages": [{"role": "user", "content": prompt}],
#             "temperature": 0.1,
#             "max_tokens": 500
#         }

#         try:
#             response = requests.post(self._base_url, headers=headers, json=data)
#             response.raise_for_status()
#             result = response.json()
#             return result["choices"][0]["message"]["content"]
#         except Exception as e:
#             return f"Error: {str(e)}"

# class PDFDocumentFAQSystem:
#     def __init__(self):
#         self.embeddings = HuggingFaceEmbeddings(
#             model_name='sentence-transformers/all-MiniLM-L6-v2'
#         )

#         groq_api_key = GROQ_API_KEY
#         if not groq_api_key or groq_api_key == 'YOUR_GROQ_API_KEY_HERE':
#             raise ValueError("Please set your GROQ API key in the GROQ_API_KEY variable!")

#         self.llm = GroqLLM(api_key=groq_api_key, model='llama3-8b-8192')

#         self.text_splitter = RecursiveCharacterTextSplitter(
#             chunk_size=1000,
#             chunk_overlap=200,
#             length_function=len
#         )

#         self.documents = []
#         self.vectorstore = None
#         self.qa_chain = None
#         self.retriever = None

#     def load_pdf(self, file_path: str) -> bool:
#         try:
#             if not os.path.exists(file_path):
#                 print(f"File not found: {file_path}")
#                 return False

#             if not file_path.lower().endswith('.pdf'):
#                 print("Please provide a PDF file.")
#                 return False

#             print("Loading PDF...")
#             loader = PyPDFLoader(file_path)
#             documents = loader.load()

#             print("Processing document...")
#             chunks = self.text_splitter.split_documents(documents)

#             print("Creating embeddings...")
#             self.vectorstore = FAISS.from_documents(chunks, self.embeddings)

#             semantic_retriever = self.vectorstore.as_retriever(search_kwargs={"k": 5})
#             texts = [doc.page_content for doc in chunks]
#             bm25_retriever = BM25Retriever.from_texts(texts)
#             bm25_retriever.k = 5

#             self.retriever = EnsembleRetriever(
#                 retrievers=[semantic_retriever, bm25_retriever],
#                 weights=[0.7, 0.3]
#             )

#             prompt_template = """
#             Use the following context to answer the question accurately.

#             Context: {context}
#             Question: {question}

#             Provide a clear, accurate answer based on the context. If the context doesn't contain enough information, say so clearly.

#             Answer:"""

#             PROMPT = PromptTemplate(
#                 template=prompt_template,
#                 input_variables=["context", "question"]
#             )

#             self.qa_chain = RetrievalQA.from_chain_type(
#                 llm=self.llm,
#                 chain_type="stuff",
#                 retriever=self.retriever,
#                 chain_type_kwargs={"prompt": PROMPT},
#                 return_source_documents=True
#             )

#             print("PDF processed successfully!")
#             return True

#         except Exception as e:
#             print(f"Error processing PDF: {str(e)}")
#             return False

#     def calculate_confidence_score(self, question: str, answer: str, source_docs: List[Document]) -> float:
#         try:
#             doc_factor = min(len(source_docs) / 5.0, 1.0)

#             if self.vectorstore and question:
#                 similar_docs = self.vectorstore.similarity_search_with_score(question, k=3)
#                 if similar_docs:
#                     avg_similarity = np.mean([1 / (1 + score) for _, score in similar_docs])
#                     similarity_factor = min(avg_similarity, 1.0)
#                 else:
#                     similarity_factor = 0.5
#             else:
#                 similarity_factor = 0.5

#             answer_length = len(answer.split())
#             length_factor = min(answer_length / 50.0, 1.0)

#             uncertainty_phrases = [
#                 "i don't know", "not sure", "unclear", "cannot determine",
#                 "insufficient information", "not enough information", "doesn't contain"
#             ]
#             uncertainty_penalty = 0.3 if any(phrase in answer.lower() for phrase in uncertainty_phrases) else 0

#             confidence = (doc_factor * 0.3 + similarity_factor * 0.4 + length_factor * 0.3) - uncertainty_penalty
#             confidence = max(0.1, min(1.0, confidence))

#             return round(confidence, 2)

#         except Exception as e:
#             print(f"Error calculating confidence: {str(e)}")
#             return 0.5

#     def ask_question(self, question: str) -> Tuple[str, float, float]:
#         if not self.qa_chain:
#             return "Please load a PDF document first.", 0.0, 0.0

#         start_time = time.time()

#         try:
#             result = self.qa_chain({"query": question})
#             processing_time = time.time() - start_time

#             answer = result['result']
#             source_docs = result.get('source_documents', [])

#             confidence = self.calculate_confidence_score(question, answer, source_docs)

#             return answer, processing_time, confidence

#         except Exception as e:
#             processing_time = time.time() - start_time
#             return f"Error: {str(e)}", processing_time, 0.0

# def main_streamlit():
#     st.set_page_config(
#         page_title="PDF FAQ System",
#         page_icon="📄",
#         layout="wide"
#     )

#     st.title("📄 PDF Document FAQ System")
#     st.markdown("Upload a PDF document and ask questions about its content!")

#     if 'faq_system' not in st.session_state:
#         try:
#             st.session_state.faq_system = PDFDocumentFAQSystem()
#             st.session_state.pdf_loaded = False
#         except ValueError as e:
#             st.error(f"Error initializing system: {e}")
#             st.stop()

#     with st.sidebar:
#         st.header("📁 Document Upload")

#         pdf_path = st.text_input("Enter PDF file path:", placeholder="/content/your_file.pdf")

#         if st.button("Load PDF") and pdf_path:
#             if not st.session_state.pdf_loaded:
#                 with st.spinner("Processing PDF... This may take a moment."):
#                     success = st.session_state.faq_system.load_pdf(pdf_path)

#                 if success:
#                     st.session_state.pdf_loaded = True
#                     st.success("✅ PDF loaded successfully!")
#                 else:
#                     st.error("❌ Failed to load PDF. Please try again.")

#     if st.session_state.pdf_loaded:
#         st.success("📖 PDF document is ready for questions!")

#         col1, col2 = st.columns([2, 1])

#         with col1:
#             question = st.text_area(
#                 "Ask a question about the PDF:",
#                 height=100,
#                 placeholder="Enter your question here..."
#             )

#             ask_button = st.button("🔍 Ask Question", type="primary")

#         with col2:
#             st.markdown("### 📊 Metrics")
#             time_placeholder = st.empty()
#             confidence_placeholder = st.empty()

#         if ask_button and question:
#             with st.spinner("Processing your question..."):
#                 answer, processing_time, confidence = st.session_state.faq_system.ask_question(question)

#             time_placeholder.metric("⏱️ Processing Time", f"{processing_time:.2f}s")
#             confidence_placeholder.metric("🎯 Confidence Score", f"{confidence:.2f}/1.00")

#             st.markdown("### 💬 Answer")

#             if confidence >= 0.8:
#                 st.success(answer)
#             elif confidence >= 0.6:
#                 st.info(answer)
#             elif confidence >= 0.4:
#                 st.warning(answer)
#             else:
#                 st.error(answer)

#             st.markdown("### 📈 Confidence Interpretation")
#             if confidence >= 0.8:
#                 st.markdown("🟢 **High Confidence** - The answer is likely very accurate based on the document content.")
#             elif confidence >= 0.6:
#                 st.markdown("🟡 **Medium Confidence** - The answer is reasonably accurate but may need verification.")
#             elif confidence >= 0.4:
#                 st.markdown("🟠 **Low Confidence** - The answer is uncertain, please verify with the original document.")
#             else:
#                 st.markdown("🔴 **Very Low Confidence** - The information may not be available in the document.")

#         elif ask_button and not question:
#             st.warning("Please enter a question first.")

#     else:
#         st.info("👈 Please enter PDF file path and click 'Load PDF' to get started!")

#         st.markdown("### 💡 How to use:")
#         st.markdown("""
#         1. **Enter PDF Path**: Use the sidebar to enter your PDF file path (e.g., /content/your_file.pdf)
#         2. **Load PDF**: Click 'Load PDF' button to process your document
#         3. **Ask Questions**: Type your questions in the text area
#         4. **Get Answers**: View answers with processing time and confidence scores
#         """)

# if __name__ == "__main__":
#     main_streamlit()

# # Cell 7: Start Streamlit with Colab Tunnel
# # !pip install pyngrok

# import subprocess
# import threading
# from pyngrok import ngrok
# import time
# import os

# # Kill any existing streamlit processes
# !pkill -f streamlit

# # Add your ngrok authtoken here
# # Replace 'YOUR_NGROK_AUTHTOKEN' with your actual authtoken from https://dashboard.ngrok.com/get-started/your-authtoken
# # It's recommended to store this as an environment variable for security in production
# # *** REPLACE 'YOUR_NGROK_AUTHTOKEN' BELOW WITH YOUR ACTUAL NGROK AUTH TOKEN ***
# NGROK_AUTHTOKEN = os.environ.get("NGROK_AUTHTOKEN", "2yJey6nB4wBpzBrxy4RHlRaTbsX_3CW538mncRysQJvFgXf8n") # Replace with your token or set environment variable
# ngrok.set_auth_token(NGROK_AUTHTOKEN)

# # Start Streamlit in background
# def run_streamlit():
#     subprocess.run(['streamlit', 'run', 'streamlit_app.py', '--server.port', '8501', '--server.headless', 'true'])

# # Start streamlit in a separate thread
# streamlit_thread = threading.Thread(target=run_streamlit)
# streamlit_thread.daemon = True
# streamlit_thread.start()

# # Wait a moment for streamlit to start
# time.sleep(10)


# # Cell 8: Create ngrok tunnel and keep alive

# # Create ngrok tunnel
# public_url = ngrok.connect(8501)
# print(f"🌐 Your Streamlit app is running at: {public_url}")
# print(f"📱 Click the link above to access your PDF FAQ System!")

# # Keep the tunnel alive
# try:
#     # This will keep the cell running and the tunnel active
#     # Wait indefinitely for the streamlit thread to finish
#     streamlit_thread.join()
# except KeyboardInterrupt:
#     print("Stopping the application...")
# finally:
#     # Ensure ngrok process is killed on exit
#     ngrok.kill()


# # Cell 9: Alternative - Direct CLI
# main_cli()

ERROR: Operation cancelled by user


UsageError: Line magic function `%%writefile` not found.
